# Preview utilities demo

Notebook de validación visual interactiva para todas las funciones públicas de `ds_utils.eda.preview_utilities`.

Cada sección incluye:
1. Una breve explicación del propósito de la función y de los elementos visuales a verificar.
2. Datos sintéticos autocontenidos y reproducibles.
3. La invocación real a la función para inspeccionar los datos resultantes.

## Imports

In [42]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ds_utils.eda.preview_config import Alignment, Color, DEFAULT_COLOR
from ds_utils.preprocessing.feature_config import FEATURES_CONFIG

# Dependencia opcional del extra EDA preview_utilities
try:
    from IPython.display import display, HTML
except ImportError as exc:
    raise ImportError(
        "This notebook requires the EDA dependencies. "
        "Install them with: uv sync --extra eda"
    ) from exc

from ds_utils.eda.preview_utilities import (
    format_time_gap,
    print_table_section,
    print_list_section,
    print_dict_section,
    print_section,
    print_console_section,
    preview_dataframe,
    show_df_details,
    show_df_differences,
    show_noise_summary,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Test data

Generamos conjuntos de datos sintéticos autocontenidos para alimentar los ejemplos visuales de las distintas funciones.

In [2]:
# ---------------------------------------------------------------------------
# 1. DataFrame tabular general para tablas y vistas previas
# ---------------------------------------------------------------------------
df_sample = pd.DataFrame({
    "id_sensor": ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"],
    "ubicacion": ["Norte", "Sur", "Este", "Oeste", "Norte", "Sur", "Este", "Oeste"],
    "temperatura": [21.5, 23.8, 19.4, 25.1, 22.0, 24.3, 18.9, 26.5],
    "humedad": [45.2, 52.0, 48.7, 61.3, 44.8, 55.1, 50.2, 63.0],
    "activo": [True, True, False, True, True, False, True, True],
})

# ---------------------------------------------------------------------------
# 2. Serie temporal con valores nulos, constantes y saltos (gaps)
# ---------------------------------------------------------------------------
# Generamos 24 horas cada 1 hora con saltos intencionados
dates = pd.date_range("2026-06-01 00:00", periods=24, freq="1h")
# Omitimos algunas horas para simular gaps (horas 5-7 y 14-17)
dates_with_gaps = dates.delete([5, 6, 7, 14, 15, 16, 17])

df_time_series = pd.DataFrame({
    "timestamp": dates_with_gaps,
    "presion": [1013.2 + i * 0.1 if i % 4 != 0 else np.nan for i in range(len(dates_with_gaps))],
    "caudal": [50.0 + i for i in range(len(dates_with_gaps))],
    "estado": ["OK" for _ in range(len(dates_with_gaps))],  # Columna constante
    "observaciones": ["" if i % 3 == 0 else f"Obs_{i}" for i in range(len(dates_with_gaps))],
})

# ---------------------------------------------------------------------------
# 3. DataFrames para comparación de diferencias estructurales
# ---------------------------------------------------------------------------
df_orig = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "fecha": ["2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04"],
    "temperatura_raw": [20.1, 21.0, 55.4, 20.8],
    "columna_obsoleta": ["A", "B", "C", "D"],
})

df_mod = pd.DataFrame({
    "id": [1, 2, 4],  # Fila 3 eliminada por filtro
    "fecha": ["2026-01-01", "2026-01-02", "2026-01-04"],
    "temperatura_raw": [20.1, 21.0, 20.8],
    "temperatura_clean": [20.1, 21.0, 20.8],
    "temperatura_is_noise": [False, False, False],
})

# ---------------------------------------------------------------------------
# 4. Datos de sensores con ruido para show_noise_summary
# ---------------------------------------------------------------------------
n_records = 50
df_noise = pd.DataFrame({
    "temperatura": 20.0 + np.random.randn(n_records) * 1.5,
    f"temperatura{FEATURES_CONFIG.IS_NOISE_SUFFIX}": [True if i in [5, 18, 33] else False for i in range(n_records)],
    "presion": 1013.0 + np.random.randn(n_records) * 3.0,
    f"presion{FEATURES_CONFIG.IS_NOISE_SUFFIX}": [True if i in [12, 40] else False for i in range(n_records)],
    FEATURES_CONFIG.GLOBAL_NOISE_COLUMN: [True if i in [5, 12, 18, 33, 40] else False for i in range(n_records)],
})
# Inyectar picos visibles en los índices anómalos
df_noise.loc[[5, 18, 33], "temperatura"] = [45.0, 52.0, -10.0]
df_noise.loc[[12, 40], "presion"] = [1050.0, 970.0]

## format_time_gap

Convierte un alias de frecuencia de Pandas (`s`, `min`, `h`, `D`, `W`) y una cantidad de periodos en una descripción legible en español con la mayor unidad temporal exacta aplicable.

In [30]:
ejemplos_freq = [
    ("s", 1),
    ("s", 45),
    ("min", 1),
    ("min", 30),
    ("h", 1),
    ("h", 12),
    ("D", 1),
    ("D", 5),
    ("W", 1),
    ("W", 7)
]

for freq, periods in ejemplos_freq:
    resultado = format_time_gap(freq, periods)
    print(f"format_time_gap(freq='{freq}', periods={periods}) -> {resultado}")

format_time_gap(freq='s', periods=1) -> 1 segundo
format_time_gap(freq='s', periods=45) -> 45 segundos
format_time_gap(freq='min', periods=1) -> 1 minuto
format_time_gap(freq='min', periods=30) -> 30 minutos
format_time_gap(freq='h', periods=1) -> 1 hora
format_time_gap(freq='h', periods=12) -> 12 horas
format_time_gap(freq='D', periods=1) -> 1 día
format_time_gap(freq='D', periods=5) -> 5 días
format_time_gap(freq='W', periods=1) -> 1 semana
format_time_gap(freq='W', periods=7) -> 7 semanas


## print_table_section

Muestra un bloque estilizado en HTML conteniendo una tabla a partir de un DataFrame, un diccionario (pares clave-valor o columnas) o una lista de registros. Permite configurar color de encabezado, alineación y pie de página.

In [ ]:
# Caso 1: DataFrame con encabezado centrado en azul y pie de página
print_table_section(
    title="DataFrame con encabezado centrado en azul y pie de página",
    content=df_sample.head(4),
    show_table_headers=True,
    show_table_index=False,
    header_color=Color.BLUE,
    header_align=Alignment.CENTER,
    footer="Datos actualizados al inicio del turno.",
)

# Caso 2: Diccionario clave-valor con encabezado naranja
metricas = {
    "Precisión": "98.4%",
    "Latencia media": "12 ms",
    "Total peticiones": "1.450.200",
    "Errores 5xx": "0",
}
print_table_section(
    title="Diccionario clave-valor con encabezado naranja",
    content=metricas,
    header_color=Color.ORANGE,
    header_align=Alignment.LEFT,
)

# Caso 3: Diccionario con listas de valores (columnas del mismo tamaño)
columnas_dict = {
    "Fase": ["Ingesta", "Limpieza", "Entrenamiento", "Evaluación"],
    "Duración (s)": [12.4, 45.1, 120.8, 8.5],
    "Resultado": ["OK", "OK", "OK", "OK"],
}
print_table_section(
    title="Diccionario con listas de valores (Columnas) y pie de página",
    content=columnas_dict,
    header_color=Color.GREEN,
    header_align=Alignment.LEFT,
    footer="Ejecutado con uv en entorno Python 3.12",
)

id_sensor,ubicacion,temperatura,humedad,activo
S1,Norte,21.5,45.2,True
S2,Sur,23.8,52.0,True
S3,Este,19.4,48.7,False
S4,Oeste,25.1,61.3,True


Key,Value
Precisión,98.4%
Latencia media,12 ms
Total peticiones,1.450.200
Errores 5xx,0


Fase,Duración (s),Resultado
Ingesta,12.4,OK
Limpieza,45.1,OK
Entrenamiento,120.8,OK
Evaluación,8.5,OK


## print_list_section

Muestra una lista de elementos formateada dentro de un bloque HTML, con opción de viñetas estándar o lista numerada (`numbered=True`).

In [35]:
reglas_negocio = [
    "Comprobar que todas las columnas requeridas existen.",
    "Identificar valores constantes antes del modelado.",
    "Reemplazar picos de ruido aislados en sensores.",
    "Validar la consistencia de la frecuencia temporal.",
]

# Caso 1: Lista sin numerar (viñetas) con color púrpura
print_list_section(
    title="Lista sin numerar centrada en color púrpura",
    items=reglas_negocio,
    header_color=Color.PURPLE,
    header_align=Alignment.CENTER,
)

# Caso 2: Lista numerada con color amarillo y pie de página
print_list_section(
    title="Lista numerada en color amarillo y pie de página",
    items=reglas_negocio,
    header_color=Color.YELLOW,
    header_align=Alignment.LEFT,
    numbered=True,
    footer="Revisar cada punto antes de promover el dataset a producción.",
)

## print_dict_section

Convierte un diccionario en líneas de texto, clave (alineado a la izquierda) y valor (alineado a la derecha), dentro de una sección diagnóstica estilizada.

In [43]:
configuracion_modelo = {
    "Algoritmo": "IsolationForest",
    "n_estimators": 150,
    "contamination": 0.05,
    "random_state": 42,
    "max_features": "sqrt (0.32)",
}

print_dict_section(
    title="Diccionario clave-valor simple",
    content=configuracion_modelo,
    header_color=Color.BLUE,
    header_align=Alignment.LEFT,
    footer="La tabla no es representativa",
)

## print_section

Renderiza un bloque HTML para mensajes diagnósticos o texto enriquecido con título, borde superior estilizado y pie opcional.

In [ ]:
mensaje_diagnostico = """El dataset cumple con todos los contratos de esquema establecidos.
- Columnas esperadas: 5 / 5 detectadas
- Tipo de datos verificado: Correcto
- Detección de duplicados: 0 duplicados encontrados"""

print_section(
    title="Sección de texto",
    content=mensaje_diagnostico,
    header_color=Color.GREEN,
    header_align=Alignment.LEFT,
    footer="Validación completada sin alertas críticas.",
)

## print_console_section

Muestra una sección en texto plano puro con separadores ASCII (`=`), sin estilos HTML, adecuada para salidas de consola o logs.

In [48]:
log_content = "Proceso batch iniciado\nLectura de 1.000 registros finalizada\nTiempo total: 0.42 s"
print_console_section(
    title="Sección de texto en Consola",
    content=log_content,
    separator_length=40
)

SECCIÓN DE TEXTO EN CONSOLA
Proceso batch iniciado
Lectura de 1.000 registros finalizada
Tiempo total: 0.42 s


## preview_dataframe

Muestra una previsualización de las primeras N filas del DataFrame con un título HTML estilizado mediante la API de estilos (`style.set_caption`).

In [52]:
# Previsualización de las primeras 5 filas con título centrado en color naranja
preview_dataframe(
    title="Previsualización del Dataframe",
    df=df_sample,
    rows=5,
    header_color=Color.ORANGE,
    header_align=Alignment.CENTER,
)

,id_sensor,ubicacion,temperatura,humedad,activo
0,S1,Norte,21.500000,45.200000,True
1,S2,Sur,23.800000,52.000000,True
2,S3,Este,19.400000,48.700000,False
3,S4,Oeste,25.100000,61.300000,True
4,S5,Norte,22.000000,44.800000,True


## show_df_details

Calcula y presenta un resumen detallado de la estructura de un DataFrame:
- Conteo de celdas y filas con valores no válidos (None, NaN, NaT, cadenas vacías).
- Detección de columnas con valores constantes.
- Comprobación de timestamps faltantes según la frecuencia (`freq_missing`).
- Con `full_info=True`: resumen de columnas (tipos, valores distintos, % nulos) y detección de gaps temporales grandes (`missing_gap_limit`).
- Con `resume_info=True`: describe estadístico e info general.

In [51]:
# Caso 1: Resumen básico con frecuencia esperada de 1 hora
show_df_details(
    df=df_time_series,
    col_time="timestamp",
    full_info=False,
    resume_info=False,
    freq_missing="1h",
)

In [54]:
# Caso 2: Inspección completa (resumen de columnas, detección de gaps >= 2 horas y describe)
show_df_details(
    df=df_time_series,
    col_time="timestamp",
    full_info=True,
    resume_info=True,
    freq_missing="1h",
    missing_gap_limit=2,
)

,nombre_columna,tipo_dato,num_valores_distintos,num_valores_no_validos,porcentaje_no_validos
0,observaciones,str,12,6,35.29
1,presion,float64,12,5,29.41
2,estado,str,1,0,0.00
3,timestamp,datetime64[us],17,0,0.00
4,caudal,float64,17,0,0.00


start,end,missing_periods,duration
2026-06-01 04:00:00,2026-06-01 08:00:00,3,0 days 04:00:00
2026-06-01 13:00:00,2026-06-01 18:00:00,4,0 days 05:00:00


,timestamp,presion,caudal
count,17,12.000000,17.000000
mean,2026-06-01 11:31:45.882353,1014.000000,58.000000
min,2026-06-01 00:00:00,1013.300000,50.000000
0.1%,2026-06-01 00:00:57.600000,1013.301100,50.016000
1%,2026-06-01 00:09:36,1013.311000,50.160000
5%,2026-06-01 00:48:00,1013.355000,50.800000
25%,2026-06-01 04:00:00,1013.650000,54.000000
50%,2026-06-01 11:00:00,1014.000000,58.000000
75%,2026-06-01 19:00:00,1014.350000,62.000000
95%,2026-06-01 22:12:00,1014.645000,65.200000


RESUMEN RAW DEL DATAFRAME

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      17 non-null     datetime64[us]
 1   presion        12 non-null     float64       
 2   caudal         17 non-null     float64       
 3   estado         17 non-null     str           
 4   observaciones  17 non-null     str           
dtypes: datetime64[us](1), float64(2), str(2)
memory usage: 812.0 bytes
None


## show_df_differences

Compara dos DataFrames (original vs modificado), reportando diferencias dimensionales (celdas, filas, columnas) y la lista de columnas añadidas y eliminadas.

In [55]:
show_df_differences(
    df_original=df_orig,
    df_modified=df_mod,
)

## show_noise_summary

Calcula estadísticas descriptivas (media, mediana, mínimo, máximo, umbral y picos detectados) para sensores analizados con columnas de ruido (según `FEATURES_CONFIG`), junto con el total global de registros con ruido.

In [56]:
# Umbrales de variación configurados por sensor
tolerancias_sensores = {
    "temperatura": 5.0,
    "presion": 10.0,
}

show_noise_summary(
    df=df_noise,
    tolerances=tolerancias_sensores,
    config=FEATURES_CONFIG,
)